In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from pangya_physics import CLUBS, Wind
from pangya_physics.ball import Ball
from pangya_physics.solver import create_initial_velocity
from pangya_physics.simulator import PangyaSimulator

In [3]:
wind = Wind(speed=0, degree=0)

targets = [100, 150, 200, 250, 300]

...

Ellipsis

In [4]:
import pandas as pd
from pangya_physics import CLUBS, Wind, find_power

wind = Wind(speed=0, degree=0)

targets = [100, 150, 200, 250, 300]

rows = []

for club_name, club in CLUBS.items():
    for target in targets:

        result = find_power(
            club=club,
            wind=wind,
            target_distance=target,
            target_height=0,
        )

        rows.append({
            "club": club_name,
            "target_distance": target,
            "power_percent": result.power_percent,
            "power_range": result.power_range,
            "final_distance": result.final_distance,
            "error": result.error,
            "iterations": result.iterations,
            "found": result.found,
        })

df_power = pd.DataFrame(rows)

df_power.head()

,club,target_distance,power_percent,power_range,final_distance,error,iterations,found
0,1W,100,0.451563,103.859375,99.680073,0.319927,8,True
1,1W,150,0.573287,131.856017,148.839671,1.160329,60,False
2,1W,200,0.683347,157.169712,200.692427,-0.692427,60,False
3,1W,250,0.790234,181.753906,250.475266,-0.475266,10,True
4,1W,300,0.896875,206.281250,299.507005,0.492995,7,True


In [5]:
from pangya_physics import CLUBS, Wind
from pangya_physics.ball import Ball
from pangya_physics.solver import create_initial_velocity
from pangya_physics.simulator import PangyaSimulator

rows = []
wind = Wind(speed=0, degree=0)

for club_name, club in CLUBS.items():
    for power in [x / 100 for x in range(10, 131)]:
        ball = Ball(
            velocity=create_initial_velocity(
                club=club,
                power_percent=power,
            )
        )

        simulator = PangyaSimulator(
            ball=ball,
            club=club,
            wind=wind,
        )

        max_height = 0

        for _ in range(1000):
            simulator.step()

            max_height = max(
                max_height,
                simulator.ball.position.y,
            )

            if simulator.ball.position.y < 0 and simulator.ball.count > 10:
                break

        rows.append({
            "club": club_name,
            "power": power,
            "distance": simulator.ball.position.z,
            "max_height": max_height,
            "flight_steps": simulator.ball.count,
        })

df_shots = pd.DataFrame(rows)

print(df_shots.shape)
df_shots.head()

(1573, 5)


,club,power,distance,max_height,flight_steps
0,1W,0.10,5.088264,0.203258,11
1,1W,0.11,6.605666,0.251600,13
2,1W,0.12,7.752369,0.302430,14
3,1W,0.13,8.988146,0.359399,15
4,1W,0.14,10.312401,0.420727,16


In [6]:
rows = []

wind = Wind(speed=0, degree=0)

for club_name, club in CLUBS.items():
    ...

df_shots = pd.DataFrame(rows)

print(df_shots.shape)
df_shots.head()

(0, 0)


""


In [7]:
df_shots.groupby("club")["distance"].max().sort_values(ascending=False)

KeyError: 'club'

In [ ]:
df_shots.groupby("club")["max_height"].max().sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt

for club in ["1W", "3W", "5I", "9I", "PW", "SW"]:
    subset = df_shots[df_shots["club"] == club]

    plt.plot(
        subset["power"],
        subset["distance"],
        label=club
    )

plt.legend()
plt.xlabel("Power")
plt.ylabel("Distance")
plt.title("Distance vs Power")
plt.show()